<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/02_scores_genre_professions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gensim pandas numpy matplotlib transformers accelerate diffusers safetensors sentencepiece

# 02 — Mesurer un axe de genre sur des professions

In [ ]:
import gensim.downloader as api
import numpy as np, pandas as pd, matplotlib.pyplot as plt
model = api.load('word2vec-google-news-300')

In [ ]:
gender_axis = model['she'] - model['he']
professions = ['neurologist','nurse','engineer','teacher','ceo','secretary','mason','housekeeper','lawyer','receptionist','surgeon','librarian']
rows=[]
for w in professions:
    if w in model:
        score = float(np.dot(model[w], gender_axis) / (np.linalg.norm(model[w]) * np.linalg.norm(gender_axis)))
        rows.append({'word':w, 'score_she_minus_he':score})
df = pd.DataFrame(rows).sort_values('score_she_minus_he')
df

In [ ]:
ax = df.plot.barh(x='word', y='score_she_minus_he', legend=False, figsize=(8,5))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Projection de professions sur l’axe she - he')
ax.set_xlabel('score : négatif --> he / positif --> she')
plt.show()

In [ ]:
import nltk
nltk.download("wordnet", quiet=True)

from nltk.corpus import wordnet as wn
import numpy as np

def descendants(synset):
    seen = set()
    stack = [synset]
    while stack:
        current = stack.pop()
        for child in current.hyponyms():
            if child not in seen:
                seen.add(child)
                stack.append(child)
    return seen

roots = [
    wn.synset("worker.n.01"),
    wn.synset("professional.n.01"),
]

job_synsets = set()
for root in roots:
    job_synsets.add(root)
    job_synsets.update(descendants(root))

candidates = set()
for syn in job_synsets:
    for lemma in syn.lemma_names():
        word = lemma.lower()
        if "_" not in word and word.isalpha() and word in model:
            candidates.add(word)

def gender_score(word):
    v = model[word]
    return float(
        np.dot(v, gender_axis) /
        (np.linalg.norm(v) * np.linalg.norm(gender_axis))
    )

auto_scores = [(word, gender_score(word)) for word in candidates]

most_she = max(auto_scores, key=lambda x: x[1])
most_he  = min(auto_scores, key=lambda x: x[1])

print(f"{len(auto_scores)} noms de métiers trouvés automatiquement dans WordNet et GNews.\n")
print(f"Profession la plus du côté de SHE : {most_she[0]}  (score = {most_she[1]:.3f})")
print(f"Profession la plus du côté de HE  : {most_he[0]}  (score = {most_he[1]:.3f})")
